# Run PreciseWikiQA Inference via Python Function

This notebook runs the equivalent of:

`scripts/run_with_server.py --step inference --task precisewikiqa --model meta-llama/Llama-3.1-8B-Instruct --N 100 --logger-type lmdb --activations-path shared/goodwiki.zarr/activations.zarr --log-file shared/goodwiki.zarr/server.log`

but through Python by calling `run_experiment(...)` from `scripts/run_with_server.py`.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "scripts" / "run_with_server.py").exists():
    raise RuntimeError("Open this notebook from the HalluLens repository root.")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.run_with_server import run_experiment

print(f"Repo root: {REPO_ROOT}")

In [ ]:
model = "meta-llama/Llama-3.1-8B-Instruct"
task = "precisewikiqa"
step = "inference"
N = 100

wiki_src = "goodwiki"
mode = "dynamic"
inference_method = "vllm"

logger_type = "lmdb"
activations_path = "shared/goodwiki.zarr/activations.zarr"
log_file = "shared/goodwiki.zarr/server.log"

host = "0.0.0.0"
port = 8000

Path(activations_path).parent.mkdir(parents=True, exist_ok=True)
print(f"Activation directory ready: {Path(activations_path).parent}")

In [ ]:
qa_output_path = f"data/precise_qa/save/qa_{wiki_src}_{model.split('/')[-1]}_{mode}.jsonl"
if not Path(qa_output_path).exists():
    raise FileNotFoundError(
        f"Missing QA file: {qa_output_path}\n"
        "Run generation first (next optional cell) or use an existing qa_output_path."
    )

print(f"Found QA file: {qa_output_path}")

## Optional: Generate Prompts First

Run the next cell only if the QA file is missing.

In [ ]:
# Uncomment to generate prompts if needed
# run_experiment(
#     step="generate",
#     task=task,
#     model=model,
#     N=N,
#     wiki_src=wiki_src,
#     mode=mode,
#     inference_method=inference_method,
#     logger_type=logger_type,
#     activations_path=activations_path,
#     log_file=log_file,
#     host=host,
#     port=port,
# )

In [ ]:
run_experiment(
    step=step,
    task=task,
    model=model,
    N=N,
    wiki_src=wiki_src,
    mode=mode,
    inference_method=inference_method,
    logger_type=logger_type,
    activations_path=activations_path,
    log_file=log_file,
    host=host,
    port=port,
)

## Outputs

- Activations: `shared/goodwiki.zarr/activations.zarr`
- Server log: `shared/goodwiki.zarr/server.log`
- Generations (default): `output/precise_wikiqa_goodwiki_dynamic/Llama-3.1-8B-Instruct/generation.jsonl`

## Natural Questions Inference (Activation Logging)

This runs Natural Questions inference and stores activation logs on disk.
`output_dir` is set to the parent directory of `activations_path` when possible (co-located outputs).

Equivalent CLI:
`python scripts/run_with_server.py --step inference --task naturalquestions --model meta-llama/Llama-3.1-8B-Instruct --inference_method vllm --N 1000 --max_tokens 64 --temperature 0.0 --logger-type zarr --activations-path shared/natural_questions_dev/activations.zarr --log-file shared/natural_questions_dev/server.log --output_dir shared/natural_questions_dev`

In [ ]:
model = "meta-llama/Llama-3.1-8B-Instruct"
task = "naturalquestions"
step = "inference"
N = 1000

inference_method = "vllm"
max_tokens = 64
temperature = 0.0

logger_type = "zarr"
activations_path = "shared/natural_questions_dev/activations.zarr"
log_file = "shared/natural_questions_dev/server.log"

data_dir = "external/LLMsKnow/data"
activations_dir = Path(activations_path).parent
output_dir = str(activations_dir) if str(activations_dir).strip() not in {"", "."} else "output"

host = "0.0.0.0"
port = 8000

activations_dir.mkdir(parents=True, exist_ok=True)
print(f"Activation directory ready: {activations_dir}")
print(f"Output directory: {output_dir}")

run_experiment(
    step=step,
    task=task,
    model=model,
    inference_method=inference_method,
    N=N,
    max_tokens=max_tokens,
    temperature=temperature,
    logger_type=logger_type,
    activations_path=activations_path,
    log_file=log_file,
    data_dir=data_dir,
    output_dir=output_dir,
    host=host,
    port=port,
)